## 🔧 Step 1: Setup and Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU availability
import torch
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not available - please enable GPU runtime")

## 📦 Step 2: Install Required Libraries

In [ ]:
# Install required packages
!pip install torch torchvision transformers
!pip install opencv-python matplotlib seaborn
!pip install scikit-learn pillow numpy pandas
!pip install pytorch-grad-cam
!pip install tqdm

print("✅ All packages installed successfully!")

## 📂 Step 3: Setup Paths and Configuration

In [ ]:
# Import libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import json
import random

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from transformers import ViTForImageClassification, ViTImageProcessor
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration for Google Drive paths
DRIVE_PATH = "/content/drive/My Drive"
DATASET_PATH = f"{DRIVE_PATH}/data"  # Your dataset folder in Drive
MODEL_SAVE_PATH = f"{DRIVE_PATH}/models"  # Where to save trained models

# Create model directory if it doesn't exist
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# Defect classes (Matching the folder names in the new dataset)
CLASS_NAMES = ['Crazing', 'Inclusion', 'Patches', 'Pitted', 'Rolled', 'Scratches']
NUM_CLASSES = len(CLASS_NAMES)

# Training configuration
IMAGE_SIZE = 224
BATCH_SIZE = 16  # Optimized for GPU
EPOCHS = 10  # Reduced for faster training
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

print("🔧 Configuration:")
print(f"   📁 Dataset Path: {DATASET_PATH}")
print(f"   💾 Model Save Path: {MODEL_SAVE_PATH}")
print(f"   🏷️ Classes: {CLASS_NAMES}")
print(f"   📐 Image Size: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"   📦 Batch Size: {BATCH_SIZE}")
print(f"   🔄 Epochs: {EPOCHS}")

## 📊 Step 4: Verify Dataset and Create Data Loaders

In [ ]:
# Verify dataset structure
def verify_dataset():
    """Verify the dataset structure in Google Drive"""
    print("🔍 Verifying dataset structure...")

    if not os.path.exists(DATASET_PATH):
        print(f"❌ Dataset path not found: {DATASET_PATH}")
        print("📝 Please ensure your dataset is uploaded to Google Drive in the 'data' folder")
        return False

    # Check for split folders
    splits = ['train', 'valid', 'test']
    total_images = 0

    for split in splits:
        split_path = os.path.join(DATASET_PATH, split)
        if not os.path.exists(split_path):
            print(f"❌ Split folder not found: {split}")
            continue

        print(f"   📂 Checking {split} split:")
        for class_name in CLASS_NAMES:
            class_path = os.path.join(split_path, class_name)
            if os.path.exists(class_path):
                image_files = [f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
                total_images += len(image_files)
                print(f"      ✅ {class_name}: {len(image_files)} images")
            else:
                print(f"      ❌ {class_name}: folder not found")

    print(f"\n📊 Total images found: {total_images}")
    return total_images > 0

# Verify dataset
dataset_ok = verify_dataset()

In [ ]:
print(f"Listing contents of: {DATASET_PATH}")
!ls -R "{DATASET_PATH}"

In [ ]:
# Verify dataset structure
def verify_dataset():
    """Verify the dataset structure in Google Drive"""
    print("🔍 Verifying dataset structure...")

    if not os.path.exists(DATASET_PATH):
        print(f"❌ Dataset path not found: {DATASET_PATH}")
        print("📝 Please ensure your dataset is uploaded to Google Drive in the 'data' folder")
        return False

    # Check for split folders
    splits = ['train', 'valid', 'test']
    total_images = 0

    for split in splits:
        split_path = os.path.join(DATASET_PATH, split)
        if not os.path.exists(split_path):
            print(f"❌ Split folder not found: {split}")
            continue

        print(f"   📂 Checking {split} split:")
        for class_name in CLASS_NAMES:
            class_path = os.path.join(split_path, class_name)
            if os.path.exists(class_path):
                image_files = [f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
                total_images += len(image_files)
                print(f"      ✅ {class_name}: {len(image_files)} images")
            else:
                print(f"      ❌ {class_name}: folder not found")

    print(f"\n📊 Total images found: {total_images}")
    return total_images > 0

# Verify dataset
dataset_ok = verify_dataset()

In [ ]:
# Custom Dataset Class
class DefectDataset(Dataset):
    """Custom PyTorch Dataset for NEU Surface Defect Classification"""

    def __init__(self, data_path, class_names, transform=None, mode='train'):
        self.data_path = data_path
        self.class_names = class_names
        self.transform = transform
        self.mode = mode

        # Create class to index mapping
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

        # Collect all image paths and labels
        self.image_paths = []
        self.labels = []

        for class_name in class_names:
            class_path = os.path.join(data_path, class_name)
            if os.path.exists(class_path):
                for filename in os.listdir(class_path):
                    if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.image_paths.append(os.path.join(class_path, filename))
                        self.labels.append(self.class_to_idx[class_name])

        print(f"📊 {mode.capitalize()} Dataset: {len(self.image_paths)} images loaded")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Convert to RGB (3 channels) for ViT
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = Image.fromarray(image)

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]
        return image, label

# Data transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Dataset class and transforms defined")

In [ ]:
# Create train/validation split and data loaders
def create_data_loaders():
    """Create training and validation data loaders"""

    if not dataset_ok:
        print("❌ Cannot create data loaders - dataset not found")
        return None, None

    # Define paths for train and validation sets
    train_path = os.path.join(DATASET_PATH, 'train')
    val_path = os.path.join(DATASET_PATH, 'valid')

    # Create datasets
    train_dataset = DefectDataset(train_path, CLASS_NAMES, transform=train_transforms, mode='train')
    val_dataset = DefectDataset(val_path, CLASS_NAMES, transform=val_transforms, mode='validation')

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"📊 Data Loaders Created:")
    print(f"   🎯 Training samples: {len(train_dataset)}")
    print(f"   🔍 Validation samples: {len(val_dataset)}")
    print(f"   📦 Batch size: {BATCH_SIZE}")

    return train_loader, val_loader

# Create data loaders
train_loader, val_loader = create_data_loaders()

In [ ]:
# ============================================================================
# ATTENTION ENTROPY LOSS - For better GradCAM visualization
# ============================================================================
import torch.nn.functional as F

class AttentionEntropyLoss(nn.Module):
    """
    Attention Entropy Loss for better explainability.
    
    This loss encourages the model's attention to be concentrated (low entropy)
    on specific regions rather than dispersed uniformly across all patches.
    
    Lower entropy = attention focuses on fewer patches = better GradCAM localization
    
    Key improvements:
    - Uses last 3 layers (more robust than just last layer)
    - Averages across attention heads
    - Temperature scaling for smooth gradients
    """
    def __init__(self, temperature=1.0, num_layers=3):
        super().__init__()
        self.temperature = temperature
        self.num_layers = num_layers
    
    def forward(self, attentions):
        # Use last N layers' attention (more robust)
        layers_to_use = attentions[-self.num_layers:]
        
        total_entropy = 0
        for layer_attention in layers_to_use:
            # Focus on CLS token's attention to image patches
            cls_attention = layer_attention[:, :, 0, 1:]  # [batch, heads, num_patches]
            
            # Average across heads for more stable training
            cls_attention_mean = cls_attention.mean(dim=1)  # [batch, num_patches]
            
            # Normalize to get probability distribution
            attn_probs = F.softmax(cls_attention_mean / self.temperature, dim=-1)
            
            # Compute entropy: H = -sum(p * log(p))
            # Lower entropy means more focused attention
            entropy = -torch.sum(attn_probs * torch.log(attn_probs + 1e-10), dim=-1)
            total_entropy += entropy.mean()
        
        # Return average entropy across layers
        return total_entropy / len(layers_to_use)

class AttentionDiversityLoss(nn.Module):
    """
    Encourages different attention heads to focus on different regions.
    This prevents all heads from looking at the same spot and promotes
    diverse feature extraction.
    """
    def __init__(self):
        super().__init__()
    
    def forward(self, attentions):
        # Use last layer
        last_layer = attentions[-1]  # [batch, heads, seq_len, seq_len]
        cls_attention = last_layer[:, :, 0, 1:]  # [batch, heads, num_patches]
        
        # Compute pairwise cosine similarity between heads
        batch_size, num_heads, num_patches = cls_attention.shape
        
        # Normalize attention
        cls_attention_norm = F.normalize(cls_attention, p=2, dim=-1)
        
        # Compute similarity matrix between heads
        # [batch, heads, patches] -> [batch, heads, heads]
        similarity = torch.bmm(cls_attention_norm, cls_attention_norm.transpose(1, 2))
        
        # We want low similarity (high diversity), so minimize the sum
        # Exclude diagonal (self-similarity)
        mask = ~torch.eye(num_heads, dtype=torch.bool, device=similarity.device)
        diversity_loss = similarity[:, mask].mean()
        
        return diversity_loss

print("✅ Attention losses defined for better GradCAM visualization")
print("   📊 AttentionEntropyLoss: Encourages focused attention")
print("   🎯 AttentionDiversityLoss: Encourages diverse head attention")

In [ ]:
# Vision Transformer Model with Attention Output Support
class ExplainableViTDefectClassifier(nn.Module):
    """
    Vision Transformer for Defect Detection with Attention Output.
    
    This enhanced version can return attention weights during training,
    which are used for the attention entropy loss.
    """

    def __init__(self, num_classes=6, model_name="google/vit-base-patch16-224-in21k"):
        super().__init__()

        # Load pre-trained ViT model
        self.vit = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )

        print(f"🤖 Loaded ViT model: {model_name}")
        print(f"🎯 Number of classes: {num_classes}")
        print("✨ Attention-guided training enabled")

    def forward(self, x, return_attention=False):
        """
        Forward pass with optional attention output.
        
        Args:
            x: Input tensor [batch, channels, height, width]
            return_attention: If True, also returns attention weights
        """
        if return_attention:
            outputs = self.vit(x, output_attentions=True)
            return outputs.logits, outputs.attentions
        else:
            outputs = self.vit(x)
            return outputs

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ExplainableViTDefectClassifier(num_classes=NUM_CLASSES)
model = model.to(device)

print(f"🔥 Model initialized on: {device}")

# Model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Parameters:")
print(f"   📈 Total parameters: {total_params:,}")
print(f"   🎯 Trainable parameters: {trainable_params:,}")


## 🤖 Step 5: Initialize Vision Transformer Model

In [ ]:
# Vision Transformer Model with Attention Output Support
class ExplainableViTDefectClassifier(nn.Module):
    """
    Vision Transformer for Defect Detection with Attention Output.
    
    This enhanced version can return attention weights during training,
    which are used for the attention entropy loss.
    """

    def __init__(self, num_classes=6, model_name="google/vit-base-patch16-224-in21k"):
        super().__init__()

        # Load pre-trained ViT model
        self.vit = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )

        print(f"🤖 Loaded ViT model: {model_name}")
        print(f"🎯 Number of classes: {num_classes}")
        print("✨ Attention-guided training enabled")

    def forward(self, x, return_attention=False):
        """
        Forward pass with optional attention output.
        
        Args:
            x: Input tensor [batch, channels, height, width]
            return_attention: If True, also returns attention weights
        """
        if return_attention:
            outputs = self.vit(x, output_attentions=True)
            return outputs.logits, outputs.attentions
        else:
            outputs = self.vit(x)
            return outputs

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ExplainableViTDefectClassifier(num_classes=NUM_CLASSES)
model = model.to(device)

print(f"🔥 Model initialized on: {device}")

# Model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Parameters:")
print(f"   📈 Total parameters: {total_params:,}")
print(f"   🎯 Trainable parameters: {trainable_params:,}")


## 🏋️ Step 6: Training Loop with Mixed Precision

In [ ]:
# Setup training components with Attention Losses
criterion = nn.CrossEntropyLoss()
attention_entropy_loss = AttentionEntropyLoss(temperature=1.0, num_layers=3)
attention_diversity_loss = AttentionDiversityLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Mixed precision training
scaler = GradScaler()

# Attention loss weights - ADJUST IF NEEDED
# Entropy: Higher = more focused attention (recommended: 0.1-0.3)
# Diversity: Encourages heads to look at different regions (recommended: 0.05-0.1)
LAMBDA_ENTROPY = 0.15  # Increased slightly for better focus
LAMBDA_DIVERSITY = 0.05  # NEW: Encourages diverse attention heads

# Warmup epochs - gradually increase attention loss weight
# This prevents early training instability
WARMUP_EPOCHS = 2

# Early stopping patience
PATIENCE = 5
patience_counter = 0

print("🔧 Training components initialized:")
print(f"   📉 Classification Loss: CrossEntropyLoss")
print(f"   🎯 Attention Entropy Loss: λ={LAMBDA_ENTROPY} (focused attention)")
print(f"   🌈 Attention Diversity Loss: λ={LAMBDA_DIVERSITY} (diverse heads)")
print(f"   🔥 Warmup: {WARMUP_EPOCHS} epochs (gradual attention loss ramp-up)")
print(f"   ⏸️ Early Stopping: Patience = {PATIENCE} epochs")
print(f"   🚀 Optimizer: AdamW (lr={LEARNING_RATE})")
print(f"   📊 Scheduler: CosineAnnealingLR")
print(f"   ⚡ Mixed precision: Enabled")
print(f"   📏 Gradient clipping: Enabled (max_norm=1.0)")

In [ ]:
# Training functions with Enhanced Attention Regularization
def train_epoch(model, train_loader, criterion, attention_entropy_loss, attention_diversity_loss,
                optimizer, scaler, device, epoch, lambda_entropy=0.15, lambda_diversity=0.05, warmup_epochs=2):
    """Train the model for one epoch with attention regularization and warmup"""
    model.train()
    total_loss = 0
    total_cls_loss = 0
    total_entropy_loss = 0
    total_diversity_loss = 0
    correct = 0
    total = 0

    # Warmup: gradually increase attention loss weight
    if epoch < warmup_epochs:
        warmup_factor = (epoch + 1) / warmup_epochs
        current_lambda_entropy = lambda_entropy * warmup_factor
        current_lambda_diversity = lambda_diversity * warmup_factor
    else:
        current_lambda_entropy = lambda_entropy
        current_lambda_diversity = lambda_diversity

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, (data, target) in enumerate(progress_bar):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass with attention
        with autocast():
            # Get predictions AND attention weights
            logits, attentions = model(data, return_attention=True)
            
            # Standard classification loss
            cls_loss = criterion(logits, target)
            
            # Attention entropy loss (encourages focused attention)
            entropy_loss = attention_entropy_loss(attentions)
            
            # Attention diversity loss (encourages diverse head attention)
            diversity_loss = attention_diversity_loss(attentions)
            
            # Combined loss with warmup
            loss = cls_loss + current_lambda_entropy * entropy_loss + current_lambda_diversity * diversity_loss

        # Mixed precision backward pass
        scaler.scale(loss).backward()
        
        # Gradient clipping for stability
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_cls_loss += cls_loss.item()
        total_entropy_loss += entropy_loss.item()
        total_diversity_loss += diversity_loss.item()
        
        pred = logits.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

        # Update progress bar
        accuracy = 100. * correct / total
        avg_loss = total_loss / (batch_idx + 1)
        avg_entropy = total_entropy_loss / (batch_idx + 1)
        gpu_memory = f'{torch.cuda.memory_allocated()/1e9:.1f}GB' if torch.cuda.is_available() else 'CPU'
        progress_bar.set_postfix({
            'Loss': f'{avg_loss:.4f}',
            'Entropy': f'{avg_entropy:.3f}',
            'Acc': f'{accuracy:.2f}%',
            'λ': f'{current_lambda_entropy:.3f}',
            'GPU': gpu_memory
        })

    num_batches = len(train_loader)
    metrics = {
        'total_loss': total_loss / num_batches,
        'cls_loss': total_cls_loss / num_batches,
        'entropy_loss': total_entropy_loss / num_batches,
        'diversity_loss': total_diversity_loss / num_batches,
        'accuracy': accuracy
    }
    return metrics

def validate_epoch(model, val_loader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    class_correct = [0] * NUM_CLASSES
    class_total = [0] * NUM_CLASSES

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            loss = criterion(outputs.logits, target)

            total_loss += loss.item()
            pred = outputs.logits.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

            # Per-class accuracy
            for i in range(target.size(0)):
                label = target[i]
                class_correct[label] += pred[i].eq(target[i]).item()
                class_total[label] += 1

    accuracy = 100. * correct / total

    # Print per-class accuracy
    print("\n📊 Per-class Validation Accuracy:")
    for i, class_name in enumerate(CLASS_NAMES):
        if class_total[i] > 0:
            class_acc = 100. * class_correct[i] / class_total[i]
            print(f"   {class_name.replace('_', ' ').title()}: {class_acc:.1f}%")

    return total_loss / len(val_loader), accuracy

print("✅ Training functions with enhanced attention regularization defined")

In [ ]:
# Main training loop with Enhanced Attention-Guided Learning
if train_loader is not None and val_loader is not None:
    print("🚀 Starting enhanced attention-guided model training...")
    print(f"⏱️ Estimated training time: {EPOCHS * 3:.0f}-{EPOCHS * 5:.0f} minutes")
    print(f"🎯 Attention loss weights: Entropy λ={LAMBDA_ENTROPY}, Diversity λ={LAMBDA_DIVERSITY}")
    print(f"🔥 Warmup: {WARMUP_EPOCHS} epochs")

    best_val_accuracy = 0
    best_model_state = None
    training_start = time.time()
    patience_counter = 0

    # Training metrics
    train_losses = []
    train_cls_losses = []
    train_entropy_losses = []
    train_diversity_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        print(f"\n📊 Epoch {epoch+1}/{EPOCHS}")
        print("-" * 50)

        # Training phase with attention losses
        train_metrics = train_epoch(
            model, train_loader, criterion, attention_entropy_loss, attention_diversity_loss,
            optimizer, scaler, device, epoch,
            lambda_entropy=LAMBDA_ENTROPY, lambda_diversity=LAMBDA_DIVERSITY,
            warmup_epochs=WARMUP_EPOCHS
        )

        # Validation phase
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)

        # Learning rate scheduling
        scheduler.step()

        # Track metrics
        train_losses.append(train_metrics['total_loss'])
        train_cls_losses.append(train_metrics['cls_loss'])
        train_entropy_losses.append(train_metrics['entropy_loss'])
        train_diversity_losses.append(train_metrics['diversity_loss'])
        train_accuracies.append(train_metrics['accuracy'])
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        # Save best model to Google Drive
        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            patience_counter = 0  # Reset patience
            # Save only the vit state dict (compatible with app.py)
            best_model_state = model.vit.state_dict().copy()
            torch.save(best_model_state, f'{MODEL_SAVE_PATH}/best_vit_defect_detector.pt')
            print(f"💾 New best model saved! Val accuracy: {val_acc:.2f}%")
        else:
            patience_counter += 1
            print(f"⏸️ No improvement ({patience_counter}/{PATIENCE})")

        # Epoch summary
        epoch_time = time.time() - epoch_start
        total_elapsed = time.time() - training_start
        remaining_time = ((total_elapsed / (epoch + 1)) * (EPOCHS - epoch - 1)) / 60

        print(f"📈 Training Losses:")
        print(f"   Total: {train_metrics['total_loss']:.4f}")
        print(f"   Classification: {train_metrics['cls_loss']:.4f}")
        print(f"   Entropy: {train_metrics['entropy_loss']:.4f} (↓ = more focused)")
        print(f"   Diversity: {train_metrics['diversity_loss']:.4f} (↓ = more diverse heads)")
        print(f"📊 Validation - Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")
        print(f"⏱️ Epoch time: {epoch_time/60:.1f}min | Remaining: {remaining_time:.1f}min")
        print(f"🔧 Current LR: {optimizer.param_groups[0]['lr']:.6f}")

        # Save checkpoint every 3 epochs
        if (epoch + 1) % 3 == 0:
            checkpoint_path = f'{MODEL_SAVE_PATH}/checkpoint_epoch_{epoch+1}.pt'
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.vit.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_metrics': train_metrics,
                'val_loss': val_loss,
                'val_accuracy': val_acc
            }, checkpoint_path)
            print(f"💾 Checkpoint saved: epoch_{epoch+1}.pt")

        # Early stopping
        if patience_counter >= PATIENCE:
            print(f"\n⏹️ Early stopping triggered after {epoch+1} epochs")
            print(f"   No improvement for {PATIENCE} consecutive epochs")
            break

    print(f"\n🎉 Training completed!")
    print(f"🏆 Best validation accuracy: {best_val_accuracy:.2f}%")
    print(f"📉 Final entropy: {train_entropy_losses[-1]:.4f} (lower = better heatmaps)")
    print(f"🌈 Final diversity: {train_diversity_losses[-1]:.4f}")
    print(f"⏱️ Total training time: {(time.time() - training_start)/60:.1f} minutes")
    print("✨ Model should now produce focused and diverse GradCAM heatmaps!")

    # Load best model
    if best_model_state:
        model.vit.load_state_dict(best_model_state)
        print(f"✅ Best model loaded for final evaluation")

    # Save training metrics
    training_history = {
        'train_losses': train_losses,
        'train_cls_losses': train_cls_losses,
        'train_entropy_losses': train_entropy_losses,
        'train_diversity_losses': train_diversity_losses,
        'train_accuracies': train_accuracies,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'best_val_accuracy': best_val_accuracy,
        'final_entropy': train_entropy_losses[-1],
        'final_diversity': train_diversity_losses[-1],
        'epochs_trained': len(train_losses),
        'early_stopped': patience_counter >= PATIENCE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'lambda_entropy': LAMBDA_ENTROPY,
        'lambda_diversity': LAMBDA_DIVERSITY,
        'warmup_epochs': WARMUP_EPOCHS
    }

    with open(f'{MODEL_SAVE_PATH}/training_history.json', 'w') as f:
        json.dump(training_history, f, indent=2)
    print(f"📊 Training history saved to Drive")

else:
    print("❌ Cannot start training - data loaders not created")

## 📊 Step 7: Final Evaluation and Visualization

In [ ]:
# Final evaluation
if train_loader is not None and val_loader is not None:
    print("📊 Final Model Evaluation")
    print("=" * 40)

    # Comprehensive evaluation
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for data, target in tqdm(val_loader, desc="Final Evaluation"):
            data, target = data.to(device), target.to(device)
            outputs = model(data)

            probabilities = F.softmax(outputs.logits, dim=1)
            predictions = outputs.logits.argmax(dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    # Calculate metrics
    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None)

    print(f"\n🎯 Final Results:")
    print(f"   📈 Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   📊 Average F1-Score: {np.mean(f1):.4f}")

    # Per-class results
    print(f"\n📋 Per-Class Results:")
    for i, class_name in enumerate(CLASS_NAMES):
        print(f"   {class_name.replace('_', ' ').title()}:")
        print(f"     Precision: {precision[i]:.3f}")
        print(f"     Recall: {recall[i]:.3f}")
        print(f"     F1-Score: {f1[i]:.3f}")
        print(f"     Support: {support[i]}")

    # Save evaluation results
    evaluation_results = {
        'accuracy': float(accuracy),
        'precision': precision.tolist(),
        'recall': recall.tolist(),
        'f1_score': f1.tolist(),
        'support': support.tolist(),
        'class_names': CLASS_NAMES
    }

    with open(f'{MODEL_SAVE_PATH}/evaluation_results.json', 'w') as f:
        json.dump(evaluation_results, f, indent=2)

    print(f"\n💾 Evaluation results saved to Drive")
    print(f"📁 Model files saved to: {MODEL_SAVE_PATH}")

In [ ]:
# Plot training progress
if 'train_losses' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    axes[0].plot(range(1, len(train_losses)+1), train_losses, 'b-', label='Training Loss', linewidth=2)
    axes[0].plot(range(1, len(val_losses)+1), val_losses, 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_title('📉 Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy plot
    axes[1].plot(range(1, len(train_accuracies)+1), train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    axes[1].plot(range(1, len(val_accuracies)+1), val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    axes[1].set_title('📈 Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{MODEL_SAVE_PATH}/training_progress.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("📊 Training progress visualization saved to Drive")

## 🔥 Step 8: Verify Attention Quality with Sample Heatmaps

Now let's generate some sample heatmaps to verify that our attention-guided training worked!

In [ ]:
# Install GradCAM if not already installed
!pip install pytorch-grad-cam -q

print("✅ GradCAM library installed")

In [ ]:
# GradCAM Heatmap Generation for Verification
from pytorch_grad_cam import GradCAM, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def generate_gradcam_heatmap(model, image_tensor, original_image, predicted_class=None):
    """Generate GradCAM heatmap for a given image"""
    
    # Define target layer (last layer of ViT encoder)
    target_layers = [model.vit.vit.encoder.layer[-1].layernorm_before]
    
    # Initialize GradCAM
    cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=None)
    
    # Generate heatmap
    targets = [ClassifierOutputTarget(predicted_class)] if predicted_class is not None else None
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0), targets=targets)
    grayscale_cam = grayscale_cam[0, :]
    
    # Overlay heatmap on original image
    visualization = show_cam_on_image(original_image, grayscale_cam, use_rgb=True)
    
    return visualization, grayscale_cam

def visualize_samples_with_heatmaps(model, val_loader, device, num_samples=6):
    """Visualize sample predictions with GradCAM heatmaps"""
    model.eval()
    
    # Get random samples
    samples = []
    for images, labels in val_loader:
        for i in range(min(num_samples - len(samples), len(images))):
            samples.append((images[i], labels[i]))
        if len(samples) >= num_samples:
            break
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (image, true_label) in enumerate(samples):
        # Prepare image
        image_tensor = image.to(device)
        
        # Get prediction
        with torch.no_grad():
            output = model(image_tensor.unsqueeze(0))
            pred_class = output.logits.argmax(dim=1).item()
            confidence = F.softmax(output.logits, dim=1)[0, pred_class].item()
        
        # Denormalize image for display
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_display = image.cpu().numpy().transpose(1, 2, 0)
        img_display = (img_display * std + mean)
        img_display = np.clip(img_display, 0, 1)
        
        # Generate GradCAM
        heatmap_vis, heatmap = generate_gradcam_heatmap(model, image_tensor, img_display, pred_class)
        
        # Plot
        axes[idx, 0].imshow(img_display)
        axes[idx, 0].set_title(f'Original\\nTrue: {CLASS_NAMES[true_label]}', fontsize=10)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(heatmap, cmap='jet')
        axes[idx, 1].set_title(f'Attention Heatmap\\nFocused regions', fontsize=10)
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(heatmap_vis)
        pred_color = 'green' if pred_class == true_label else 'red'
        axes[idx, 2].set_title(f'Overlay\\nPred: {CLASS_NAMES[pred_class]}\\nConf: {confidence:.2%}', 
                               fontsize=10, color=pred_color)
        axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{MODEL_SAVE_PATH}/sample_heatmaps.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Sample heatmaps saved to: {MODEL_SAVE_PATH}/sample_heatmaps.png")
    print("\\n🎯 Check if attention is focused on defect regions!")
    print("   Good signs: Bright spots on actual defects, not scattered across image")

print("✅ GradCAM visualization functions defined")

In [ ]:
# Generate sample heatmaps to verify attention quality
if val_loader is not None and 'model' in locals():
    print("🔥 Generating sample heatmaps to verify attention-guided training...")
    visualize_samples_with_heatmaps(model, val_loader, device, num_samples=6)
else:
    print("❌ Model or validation loader not available")

In [ ]:
# Analyze attention entropy distribution
if 'train_entropy_losses' in locals() and len(train_entropy_losses) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Entropy trend
    axes[0].plot(range(1, len(train_entropy_losses)+1), train_entropy_losses, 
                 'b-', linewidth=2, marker='o', markersize=6)
    axes[0].set_title('🎯 Attention Entropy Over Training\\n(Lower = More Focused)', 
                      fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Entropy')
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(y=train_entropy_losses[-1], color='r', linestyle='--', 
                    label=f'Final: {train_entropy_losses[-1]:.3f}')
    axes[0].legend()
    
    # Diversity trend
    if 'train_diversity_losses' in locals():
        axes[1].plot(range(1, len(train_diversity_losses)+1), train_diversity_losses, 
                     'g-', linewidth=2, marker='s', markersize=6)
        axes[1].set_title('🌈 Attention Diversity Over Training\\n(Lower = More Diverse Heads)', 
                          fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Diversity Loss')
        axes[1].grid(True, alpha=0.3)
        axes[1].axhline(y=train_diversity_losses[-1], color='r', linestyle='--', 
                        label=f'Final: {train_diversity_losses[-1]:.3f}')
        axes[1].legend()
    
    # Combined view
    axes[2].plot(range(1, len(val_accuracies)+1), val_accuracies, 
                 'r-', linewidth=2, marker='D', markersize=6, label='Val Accuracy')
    ax2_twin = axes[2].twinx()
    ax2_twin.plot(range(1, len(train_entropy_losses)+1), train_entropy_losses, 
                  'b--', linewidth=2, alpha=0.7, label='Entropy')
    axes[2].set_title('📊 Accuracy vs Attention Quality', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Validation Accuracy (%)', color='r')
    ax2_twin.set_ylabel('Attention Entropy', color='b')
    axes[2].tick_params(axis='y', labelcolor='r')
    ax2_twin.tick_params(axis='y', labelcolor='b')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(loc='upper left')
    ax2_twin.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig(f'{MODEL_SAVE_PATH}/attention_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\\n📊 Attention Quality Analysis:")
    print(f"   Initial Entropy: {train_entropy_losses[0]:.4f}")
    print(f"   Final Entropy: {train_entropy_losses[-1]:.4f}")
    print(f"   Improvement: {((train_entropy_losses[0] - train_entropy_losses[-1]) / train_entropy_losses[0] * 100):.1f}%")
    print(f"   ✅ Lower entropy = Better focused attention = Better heatmaps!")
    
    if train_entropy_losses[-1] < train_entropy_losses[0]:
        print(f"   ✨ SUCCESS! Attention became more focused during training")
    else:
        print(f"   ⚠️ Warning: Attention did not improve. Consider:")
        print(f"      - Increasing lambda_entropy (current: {LAMBDA_ENTROPY})")
        print(f"      - Training for more epochs")
        print(f"      - Adjusting learning rate")

## ✅ Training Complete!

### 🎉 **Success!** Your Vision Transformer model has been trained and saved to Google Drive.

### 📁 **Files Saved to Your Drive:**
- `best_vit_defect_detector.pt` - Best model weights
- `checkpoint_epoch_*.pt` - Training checkpoints  
- `training_history.json` - Training metrics
- `evaluation_results.json` - Final evaluation results
- `training_progress.png` - Training visualization

### 🔄 **Next Steps:**
1. **Download the trained model** from your Google Drive
2. **Use it in your local notebook** for explainability features
3. **Deploy with Streamlit** for interactive defect detection

### 📊 **Model Performance:**
The model should achieve **85-95% accuracy** on the NEU Surface Defect Dataset, ready for production use with explainable AI features!